In [ ]:
import pickle

file_path = "NewResults/GB_BO_results_6_1.0_6_72_33027.pkl"

In [ ]:
print(f"Loading results from: {file_path}\n")

with open(file_path, "rb") as f:
    results = pickle.load(f)

best_x = results["best_x"]
best_y = results["best_y"]
history = results["evaluation_history"]

print(f"Best Score (y): {best_y}")
print(f"Binary Vector:\n{best_x}\n")

In [ ]:
if len(history) > 10:
    print(f"First 5 scores: {[round(num, 4) for num in history[:5]]}")
    print(f"Last 5 scores:  {[round(num, 4) for num in history[-5:]]}")
else:
    print(history)

In [ ]:
from code_construction.code_construction import CodeConstructor, CSSCode
import codedistance

cc = CodeConstructor(method="bb", para_dict={"l": 12, "g": 6})
cc2 = CodeConstructor(method="gb", para_dict={"l":72})


def get_nkd(code: CSSCode):
    res = codedistance.CSScodeDistance(
        code.hx,
        code.hz,
        method="QDistEvol",
        params={},
        seed=100,
    )
    dist = res["d"]
    return code.n, code.k, dist

In [ ]:
from bayesian_optimization.objective_function import ObjectiveFunction

of1 = ObjectiveFunction(
    cc,
    lambda_=1,
    pp=0.05,
    decoder_param={"trail": 100_000, "max_error": 1000},
)
of2 = ObjectiveFunction(
    cc2,
    lambda_=1,
    pp=0.05,
    decoder_param={"trail": 100_000, "max_error": 1000},
)

# of_cl = ObjectiveFunction(
#     cc,
#     lambda_=1,
#     pp=0.005,
#     circuit_level_noise=True,
#     decoder_param={"trail": 100_000, "max_error": 1000},
# )

pp_list = [
    0.05,
    0.040036870145840404,
    0.032059019421497734,
    0.0256708559516296,
    0.020555614525359374,
    0.01645964939039528,
    0.013179856905786339,
    0.010553604389554513,
    0.008450665770303305,
    0.0067667641618306355,
]

gross_code = [
    0.0,
    0.0,
    0.0,
    1.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    1.0,
    1.0,
    0.0,
    0.0,
    0.0,
    0.0,
    1.0,
    1.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
    1.0,
    0.0,
    0.0,
]


In [ ]:
code = cc.construct(gross_code)

n, k, d = get_nkd(code)
print(f"[[{n}, {k}, {d}]]")

ler_plq = of1.lerpq(gross_code)
print(f"LER per logical qubit: {ler_plq}")

In [ ]:
import numpy as np

def get_row_column_weights(H):
    row_weights = np.sum(H != 0, axis=1)
    max_row_w = np.max(row_weights)
    avg_row_w = np.mean(row_weights)

    col_weights = np.sum(H != 0, axis=0)
    max_col_w = np.max(col_weights)
    avg_col_w = np.mean(col_weights)

    return max_row_w, avg_row_w, max_col_w, avg_col_w


def percent_diff(v1, v2):
    return ((v2 - v1) / v1) * 100

In [ ]:
files = [
    None,
    "NewResults\handpicked\GB_BO_results_0_0.5_4_72_10246.pkl",
    "NewResults\handpicked\GB_BO_results_1_0.5_4_72_8595.pkl",
    "NewResults\handpicked\GB_BO_results_4_0.5_6_72_16549.pkl",
    "NewResults\handpicked\GB_BO_results_11_0.5_8_72_526394.pkl",
]

ler_results = []

for file in files:
    if file is None:
        code = cc.construct(gross_code)
        best_x = gross_code
        best_y = 0
        of = of1
    else:
        with open(file, "rb") as f:
            results = pickle.load(f)
            best_x = results["best_x"]
            best_y = results["best_y"]

        best_x = best_x.cpu()
            
        if "GB" in file:
            code = cc2.construct(best_x)
            of = of2
        else:
            code = cc.construct(best_x)
            of = of1

    n, k, d = get_nkd(code)

    of.pp = pp_list[0]
    ler = of.ler(code)
    ler_pq = of.lerpq(best_x)
    
    # th, pl = of.psuedo_t(code)

    print(f"from {file}, score: {best_y:.5f}")
    print(f"[[{n}, {k}, {d}]]")
    print(f"LER per logical qubit (pp=0.05): {ler_pq}")
    mrx, arx, mcx, acx = get_row_column_weights(code.hx)
    mrz, arz, mcz, acz = get_row_column_weights(code.hz)
    print(f"Hx: max row weight={mrx}, avg row weight={arx}")
    print(f"Hx: max col weight={mcx}, avg col weight={acx}")
    print(f"Hz: max row weight={mrz}, avg row weight={arz}")
    print(f"Hz: max col weight={mcz}, avg col weight={acz}")
    # print(f"pseduo-t: h-hat={th}, pL={pl}")
    print("--------------------------------------------")

    res = []
    res.append(ler_pq)
    with open("line_res_gb.txt", 'a') as f:
        f.write(f"\n[[{n}, {k}, {d}]], {best_y:.5f}\n")
        f.write(f"{ler_pq}\n")

    for i, p in enumerate(pp_list[1:]):
        of.pp = p
        ler_pq = of.lerpq(best_x)
        res.append(ler_pq)
        with open("line_res_gb.txt", 'a') as f:
            f.write(f"{ler_pq}\n")

    ler_results.append(res)


In [ ]:
import re

# above code often got interrupted, so this function loads ler_results from a file
def parse_line_res(file_path):
    blocks = []
    current_block = []
    current_header = None
    block_start_pattern = re.compile(r"^\s*(\[\[.*?\]\])")
    
    with open(file_path, 'r') as file:
        for line in file:
            line = line.strip()
            if not line:
                continue
                
            match = block_start_pattern.match(line)
            if match:
                if current_header and current_block:
                    blocks.append({'label': current_header, 'data': current_block})
                    current_block = []

                current_header = match.group(1)
                continue
            
            try:
                val = float(line)
                current_block.append(val)
            except ValueError:
                print(f"Skipping unparseable line: {line}")
                
        if current_header and current_block:
            blocks.append({'label': current_header, 'data': current_block})
        
    return blocks

In [ ]:
import matplotlib.pyplot as plt

ler_results = parse_line_res("line_res_gb.txt")

for i, r in enumerate(ler_results):
    if i == 0:
        label = r['label'] + " Gross code"
    else:
        label = "GB " + r['label']
    plt.plot(pp_list, r['data'], label=label)

plt.xlabel('Physical Error Rate', fontsize=12)
plt.ylabel('Logical Error Rate per Logical Qubit', fontsize=12)

plt.yscale("log")
plt.xscale("log")
plt.legend()
plt.show()

In [ ]:
import time

of_default = ObjectiveFunction(
    cc,
    lambda_=1,
    pp=0.05,
)

codes = [
    gross_code,
]

for best_x in codes:
    code = cc.construct(best_x)  

    t0 = time.time()
    n, k, d = get_nkd(code)
    t1 = time.time()
    print(f"QDistEvol time: {t1-t0}")

    t2 = time.time()
    ler = of_default.ler(code)
    t3 = time.time()
    print(f"LER simulation time: {t3-t2}")